# Text Summarization: Specialist and Instruction-Tuned Models

This notebook introduces two complementary approaches:

1. **Specialist summarization:** BART and DistilBART, which are trained for news summarization.
2. **Instruction-tuned summarization:** FLAN-T5, which accepts natural-language task instructions.

Next, evaluate summaries against reference summaries using **ROUGE**, summarize **50 news articles**, and inspect the best and worst examples by ROUGE-Lsum.

**Transformers v5 note:** The old `pipeline("summarization")` task is unavailable in v5. We use `AutoModelForSeq2SeqLM.generate()` directly, through a small `summarize()` function.

Chat models are covered separately in the text-generation lesson.

### Runtime

- Pick a **GPU** runtime

### Libraries

**`rouge_score`** — computes the ROUGE metric (Recall-Oriented Understudy for Gisting
Evaluation), the standard for scoring summary quality against references.

In [1]:
%pip install -q -U "transformers>=5,<6" accelerate sentencepiece datasets rouge_score

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

Using device: cuda


### Summarization with encoder–decoder models

Our `summarize()` function loads a pretrained encoder–decoder model, tokenizes the inputs, generates summaries, and decodes them. It returns the familiar `[{'summary_text': ...}]` format.

The model is cached so that summarizing 50 articles does **not** reload the model 50 times. Changing the model name loads a different model.

In [3]:
from functools import lru_cache

@lru_cache(maxsize=1)
def load_summarization_model(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(DEVICE).eval()
    return tokenizer, model

def summarize(texts, model_name="facebook/bart-large-cnn"):
    tokenizer, model = load_summarization_model(model_name)
    texts = [texts] if isinstance(texts, str) else list(texts)

    inputs = tokenizer(
        texts, return_tensors="pt", padding=True, truncation=True
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(**inputs)

    return [
        {"summary_text": summary.strip()}
        for summary in tokenizer.batch_decode(outputs, skip_special_tokens=True)
    ]

In [4]:
content = """
  We are on the brink of a technological revolution that could jumpstart productivity, boost global growth and raise incomes around the world.
  Yet it could also replace jobs and deepen inequality. The rapid advance of artificial intelligence has captivated the world, causing both excitement and alarm, and
  raising important questions about its potential impact on the global economy. The net effect is difficult to foresee, as AI will ripple through economies in complex ways.
  What we can say with some confidence is that we will need to come up with a set of policies to safely leverage the vast potential of AI for the benefit of humanity.

  In a new analysis, IMF staff examine the potential impact of AI on the global labor market. Many studies have predicted the likelihood that jobs will be replaced by AI.
  Yet we know that in many cases AI is likely to complement human work. The IMF analysis captures both these forces.

  The findings are striking: almost 40 percent of global employment is exposed to AI. Historically, automation and information technology have tended to affect routine tasks,
  but one of the things that sets AI apart is its ability to impact high-skilled jobs. As a result, advanced economies face greater risks from AI-but also more opportunities to
  leverage its benefits-compared with emerging market and developing economies.

  In advanced economies, about 60 percent of jobs may be impacted by AI. Roughly half the exposed jobs may benefit from AI integration, enhancing productivity. For the other
  half, AI applications may execute key tasks currently performed by humans, which could lower labor demand, leading to lower wages and reduced hiring. In the most extreme
  cases, some of these jobs may disappear.

  In emerging markets and low-income countries, by contrast, AI exposure is expected to be 40 percent and 26 percent, respectively. These findings suggest emerging market and
  developing economies face fewer immediate disruptions from AI. At the same time, many of these countries don't have the infrastructure or skilled workforces to harness the
  benefits of AI, raising the risk that over time the technology could worsen inequality among nations.
"""

print(content)


  We are on the brink of a technological revolution that could jumpstart productivity, boost global growth and raise incomes around the world.
  Yet it could also replace jobs and deepen inequality. The rapid advance of artificial intelligence has captivated the world, causing both excitement and alarm, and
  raising important questions about its potential impact on the global economy. The net effect is difficult to foresee, as AI will ripple through economies in complex ways.
  What we can say with some confidence is that we will need to come up with a set of policies to safely leverage the vast potential of AI for the benefit of humanity.

  In a new analysis, IMF staff examine the potential impact of AI on the global labor market. Many studies have predicted the likelihood that jobs will be replaced by AI.
  Yet we know that in many cases AI is likely to complement human work. The IMF analysis captures both these forces.

  The findings are striking: almost 40 percent of global emp

### `facebook/bart-large-cnn`

On the Hub: Models → filter by **Summarization** → `facebook/bart-large-cnn` is a
top result. https://huggingface.co/facebook/bart-large-cnn

---

## Instruction-tuned summarization with FLAN-T5

BART-CNN is **specialized for summarization**, whereas FLAN-T5 is trained to follow natural-language instructions across tasks.

| Model | Architecture | How summarization is specified |
|---|---|---|
| `facebook/bart-large-cnn` | Encoder–decoder, summarization-specific | Load the summarization checkpoint and generate a summary |
| `google/flan-t5-base` | Instruction-tuned encoder–decoder | Provide an instruction such as “Summarize this article in two sentences” |

Use the same article to demonstrate how changing a FLAN-T5 instruction can affect format and audience. Prompt constraints are requests, not guarantees, so check the generated text manually.

In [5]:
article = content.strip()
bart_summary = summarize(article)[0]["summary_text"]

print("SPECIALIST: BART-LARGE-CNN\n")
print(bart_summary)

config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

SPECIALIST: BART-LARGE-CNN

The rapid advance of artificial intelligence has captivated the world. IMF staff examine the potential impact of AI on the global labor market. In advanced economies, about 60 percent of jobs may be impacted by AI. In emerging markets and low-income countries, by contrast, AI exposure is expected to be 40 percent.


### FLAN-T5: the instruction is part of the input

FLAN-T5 was instruction-tuned on many tasks. Unlike BART-CNN, the **same checkpoint** can be prompted to summarize, explain, classify, or reformat text.

FLAN-T5-base is small enough for CPU experiments, though a GPU is faster. Its limited input window makes it less suitable for very long documents without truncation or chunking.

Model card: https://huggingface.co/google/flan-t5-base


In [6]:
FLAN_ID = "google/flan-t5-base"

flan_tokenizer = AutoTokenizer.from_pretrained(FLAN_ID)
flan_model = AutoModelForSeq2SeqLM.from_pretrained(FLAN_ID)
flan_model.to(DEVICE)

def flan_summarize(article_text, instruction):
    prompt = f"{instruction}\n\nArticle:\n{article_text}\n\nSummary:"

    inputs = flan_tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=512
    ).to(DEVICE)

    with torch.inference_mode():
        output = flan_model.generate(
            **inputs, max_new_tokens=120, num_beams=4
        )

    return flan_tokenizer.decode(output[0], skip_special_tokens=True).strip()

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [7]:
flan_instruction = (
    "Summarize the following article in two concise sentences. "
    "Use only facts stated in the article."
)

flan_summary = flan_summarize(article, flan_instruction)
print("INSTRUCTION-TUNED: FLAN-T5\n")
print(flan_summary)

INSTRUCTION-TUNED: FLAN-T5

The rapid advance of artificial intelligence has captivated the world, causing both excitement and alarm, and raising important questions about its potential impact on the global economy. The net effect is difficult to foresee, as AI will ripple through economies in complex ways. What we can say with some confidence is that we will need to come up with a set of policies to safely leverage the vast potential of AI for the benefit of humanity. In a new analysis, IMF staff examine the potential impact of AI on the global labor market. Many studies have predicted the likelihood that jobs will be replaced


In [8]:
# Change the instruction
flan_bullets = flan_summarize(
    article,
    "Summarize the article as three short bullet points for a high-school student. "
    "Use only information in the article."
)

print("FLAN-T5, DIFFERENT INSTRUCTION\n")
print(flan_bullets)

FLAN-T5, DIFFERENT INSTRUCTION

In a new analysis, the IMF examines the potential impact of AI on the global labor market.


In [9]:
import pandas as pd

comparison = pd.DataFrame({
    "Model": ["BART-CNN (specialist)", "FLAN-T5 (instruction-tuned)"],
    "Summary": [bart_summary, flan_summary],
})

pd.set_option("display.max_colwidth", 400)
display(comparison)

,Model,Summary
0,BART-CNN (specialist),"The rapid advance of artificial intelligence has captivated the world. IMF staff examine the potential impact of AI on the global labor market. In advanced economies, about 60 percent of jobs may be impacted by AI. In emerging markets and low-income countries, by contrast, AI exposure is expected to be 40 percent."
1,FLAN-T5 (instruction-tuned),"The rapid advance of artificial intelligence has captivated the world, causing both excitement and alarm, and raising important questions about its potential impact on the global economy. The net effect is difficult to foresee, as AI will ripple through economies in complex ways. What we can say with some confidence is that we will need to come up with a set of policies to safely leverage the ..."


---

## News data and ROUGE

This section uses your original news dataset to evaluate DistilBART summaries against reference summaries. We first compare DistilBART and FLAN-T5 on one article, then restore the original **50-article evaluation** and identify the best and worst results by ROUGE-Lsum.

ROUGE measures lexical overlap with the reference, not factual accuracy or instruction-following.

### Load the news dataset

https://drive.google.com/drive/u/0/folders/11AKRIT_VEkg_f3XsIVNik6w-YB6FZj8Z

In [10]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [11]:
import os
import itertools
import pandas as pd

CSV_PATH = "/content/drive/MyDrive/oreilly_news_articles/news_articles.csv"

news_df = pd.read_csv(CSV_PATH)

news_df.sample(5)

,Content,Summary,Dataset
3098,"By . Matt Chorley, Mailonline Political Editor . The majority of babies will be born out of wedlock within three years, new figures suggest. The decline in marriage has led to a huge change in the way children are brought up in a generation. A record 47.5 per cent of babies were born to unmarried mothers in 2012, compared to just 8.4 per cent 40 years earlier. Record rise: Dramatic social chan...","Last year a record 47.5 per cent of babies were born to unmarried mother .\nIf trend continues children with marries parents will be in minority by 2016 .\n724,000 babies born in England and Wales in 2012 - highest since 1971 .",CNN/Daily Mail
4642,"England's cricketers may have slumped to a one-day international series defeat in Sri Lanka, but the team spirit seems to remain undamaged. That's judging by a tweet from batsman Ravi Bopara, who gave his team-mates the chance to play a prank on him when he forgot to lock his hotel room door. His fellow England players wasted no time in making Bopara regret the error of his ways by dumping the...",England are approaching the end of a seven-match tour of Sri Lanka .\nThe hosts have an unassailable 4-2 lead with only one match remaining .\nBut morale appears to be high among the members of the England camp .,CNN/Daily Mail
3877,"By . Arthur Martin . PUBLISHED: . 18:08 EST, 1 July 2013 . | . UPDATED: . 02:57 EST, 2 July 2013 . The number of sickening child abuse images found on the internet has doubled in a year, a report has found. As Britain's internet giants face growing demands to do more to block indecent images, experts said 70,000 pictures and videos of children being abused were discovered last year - double th...","CEOP study finds 70% rise in number of images with girls under 10 .\nAnd 125% rise in number of online images featuring sex with children .\nInternet giants facing demands to do more to block indecent images .\nLive streaming for cash is big problem, particularly in developing world .\nOffenders pay vulnerable families to film their children being abused .",CNN/Daily Mail
4437,"Southampton have made enquiries for Porto prospect Juan Quintero. Ronald Koeman is looking to take the Colombian on loan as injuries mount up at St Mary's. Quintero, 21, can play as an attacking midfielder, or on the left. He has 11 caps for his country. Colombia's Juan Quintero celebrates scoring against the Ivory Coast during the 2014 World Cup in Brazil . Ronald Koeman recently revealed tha...",Juan Quintero is an attacking midfielder and can also operate on the left .\nThe Colombian has 11 caps for his country .\nSouthampton are currently fourth in the Premier League table .\nKoeman's side travel to Old Trafford to face Manchester United on Sunday .,CNN/Daily Mail
1720,"(CNN) -- Able to scare up a few bucks? Just in time for Halloween, the owners of the home made famous in the spooky 1970s film ""The Amityville Horror"" are dramatically slashing the asking price on their Toms River, New Jersey, colonial, which was used for exteriors in the film. The decision wasn't triggered by demonic activity, however. Apparently, another frightening ""D"" word was to blame. ""M...","House used for exteriors of ""Amityville Horror"" movie is for sale .\nHouse ""not haunted, not at all,"" says Odalys Fragoso; she and husband are divorcing .\nFilm of ""The Amityville Horror"" came out in 1979, spawned several sequels .\nReal-life house that spooked the Lutz family is in Long Island .",CNN/Daily Mail


In [12]:
news_df.shape

(5000, 3)

Null value check

In [13]:
import re

def clean_txt(txt):
    txt = txt.lower()

    # Remove backslashes, @ symbols
    txt = txt.replace("\\", " ")
    txt = txt.replace("@", " ")

    # Replace non-breaking space (used to help not split words if text is wrapped)
    txt = txt.replace("\xa0", " ")

    txt = txt.replace("/", " ")
    txt = txt.replace("\n", " ")
    txt = re.sub(r"'s\b", " ", txt)
    txt = txt.replace('"', ' ')

    # Remove extra whitespace
    txt = re.sub(r'\s+', ' ', txt).strip()

    return txt

In [14]:
cleaned_df = news_df.copy()

cleaned_df['Content'] = cleaned_df['Content'].map(clean_txt)
cleaned_df['Summary'] = cleaned_df['Summary'].map(clean_txt)

cleaned_df.sample(2)

,Content,Summary,Dataset
4532,"a grandmother and her four-year-old granddaughter have drowned after they went to a lake to feed the ducks. relatives suspect that laura zulema fleming, 59, jumped into the water at silver lake, a gated community in miramar, florida, on sunday after her granddaughter leila fell in, wsvn reported. but neither could swim and they both drowned. family members called police around 11pm on sunday t...",laura fleming and granddaughter leila failed to return home 4 hours after they went to feed the ducks in their gated community in florida on sunday . their bodies were recovered from the water around midnight . relatives suspect that the child had fallen in the water and then laura - who could not swim - tried to save her .,CNN/Daily Mail
2643,"the jamaican sprinter victory brought in the highest audience of live sporting action so far this olympics. between 21:50 and 21:55 on sunday, 19.4 million viewers watched on bbc one as bolt won his second 100m gold. another 628,000 viewers saw the final on the dedicated games channel bbc olympics 3, while a further 66,000 watched in 3d on the bbc hd channel. the final was the culmination of a...","usain bolt win in the men 100 metre final was watched by more than 20 million viewers, the bbc has confirmed.",XSum


Checking an instance of Content and Summary

In [15]:
news_df['Content'][15]

"Ashton Kutcher has entered the firestorm surrounding under-fire taxi-hiring app Uber and defended controversial comments made by an executive who\xa0suggested spending $1 million to dig up dirt on journalists who criticize the company. Kutcher, an investor in the app, took to Twitter on Wednesday to show his support for beleaguered VP Emil Michael and described Sarah Lacy, a female journalist who has been highly critical of the company, as 'shady'. 'What is so wrong about digging up dirt on shady journalist?' tweeted the celebrity tech entrepreneur who has invested in tech firms including Skype, Foursquare, Airbnb and Spotify through his\xa0venture capital firm A-Grade. Under-fire: Ashton Kutcher has entered the firestorm surrounding taxi-hiring app Uber and defended Senior VP Emil Michael who suggested spending $1 million to dig up dirt on journalists who criticize the company . Outburst: Kutcher, an investor in the app, took to Twitter on Wednesday to attack Sarah Lacy, the journali

`\xa0`, `@` are removed

In [16]:
cleaned_df['Content'][15]

"ashton kutcher has entered the firestorm surrounding under-fire taxi-hiring app uber and defended controversial comments made by an executive who suggested spending $1 million to dig up dirt on journalists who criticize the company. kutcher, an investor in the app, took to twitter on wednesday to show his support for beleaguered vp emil michael and described sarah lacy, a female journalist who has been highly critical of the company, as 'shady'. 'what is so wrong about digging up dirt on shady journalist?' tweeted the celebrity tech entrepreneur who has invested in tech firms including skype, foursquare, airbnb and spotify through his venture capital firm a-grade. under-fire: ashton kutcher has entered the firestorm surrounding taxi-hiring app uber and defended senior vp emil michael who suggested spending $1 million to dig up dirt on journalists who criticize the company . outburst: kutcher, an investor in the app, took to twitter on wednesday to attack sarah lacy, the journalist who

In [17]:
news_df['Summary'][15]

"Uber investor Kutcher has tweeted his support for the under-fire app and accused critic Sarah Lacy of being 'shady'\nKutcher's comments follow the firestorm that\xa0erupted\xa0after senior VP Emil\xa0Michael\xa0suggested Uber should hire a $1 million team of researchers .\nThey 'would dig up dirt on journalists' personal lives and their families'\nMichael was reportedly speaking with specific reference to Lacy, an\xa0outspoken critic of the online taxi service .\nActor quickly apologized but was strongly criticized by Twitter users - many of whom accused him of only getting involved because he is an investor .\nIn 2011 he was forced into an embarrassing climbdown after tweeting that the sacking of Joe Paterno as head coach of Penn State showed 'no class'"

New line characters are removed

In [18]:
cleaned_df['Summary'][15]

"uber investor kutcher has tweeted his support for the under-fire app and accused critic sarah lacy of being 'shady' kutcher comments follow the firestorm that erupted after senior vp emil michael suggested uber should hire a $1 million team of researchers . they 'would dig up dirt on journalists' personal lives and their families' michael was reportedly speaking with specific reference to lacy, an outspoken critic of the online taxi service . actor quickly apologized but was strongly criticized by twitter users - many of whom accused him of only getting involved because he is an investor . in 2011 he was forced into an embarrassing climbdown after tweeting that the sacking of joe paterno as head coach of penn state showed 'no class'"

### `sshleifer/distilbart-cnn-12-6`

https://huggingface.co/sshleifer/distilbart-cnn-12-6

#### BART (Bidirectional and Auto-Regressive Transformers)
BART is an encoder–decoder model from Facebook AI. The encoder reads the input
bidirectionally (BERT-like) and the decoder generates the output autoregressively
(GPT-like), which makes it strong for sequence generation tasks like summarization,
translation, and text generation.

#### DistilBART
DistilBART is a distilled (smaller, faster) version of BART: a compact "student" model
trained to mimic a larger "teacher," keeping most of the quality while using far fewer
resources — handy for limited-hardware or real-time use.

Docs for the generation API:
https://huggingface.co/docs/transformers/main/en/main_classes/text_generation

In [19]:
example_text = cleaned_df['Content'][8]

example_text

"many people might associate roses with romance novels, but in fact the smell of chocolate tempts readers to buy loved-up fiction. researchers at antwerp university in belgium found that while a chocolatey smell makes people hungry to buy romance novels, it does not necessarily tempt people to buy grittier genres such as crime or business books. the scientists, who permeated a bookshop with the perfume of chocolate, also found that the scent encouraged customers to browse through titles. the government funded study found that bookshop visitors were almost six times more likely to buy a romance novel if they smelt chocolate - and the same results applied to cookery books too . they studied the behaviour of 201 customers in a popular chain bookshop over ten days to find that customers were 3.5 times more likely to pick up romance novels when they smelt chocolate, the guardian reported. the government funded study also found that bookshop visitors were almost six times more likely to buy 

In [20]:
summary_txt = summarize(example_text, model_name="sshleifer/distilbart-cnn-12-6")

summary_txt

config.json:   0%|          | 0.00/1.80k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.22GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.22GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

[{'summary_text': 'Researchers at antwerp university in belgium, Belgium, permeated a bookshop with the perfume of chocolate . They found customers were 3.5 times more likely to buy romance novels when they smelt chocolate . The government funded study also found the scent encouraged customers to browse through titles .'}]

In [21]:
summary_txt = clean_txt(summary_txt[0]['summary_text'])

summary_txt

'researchers at antwerp university in belgium, belgium, permeated a bookshop with the perfume of chocolate . they found customers were 3.5 times more likely to buy romance novels when they smelt chocolate . the government funded study also found the scent encouraged customers to browse through titles .'

In [22]:
ref_txt = cleaned_df['Summary'][8]

ref_txt

'belgium scientists flooded a bookshop with a chocolatey smell to find customers were almost six times more likely to buy a romance novel . the government-funded study said the scent does not have a big effect on making people want to purchase travel, crime, business or comic books . researchers at antwerp university said the sweet smell encouraged customers to browse and . sales of all books rose during the experiment .'

### Scoring summaries with ROUGE

ROUGE compares a generated summary to one or more reference summaries.

- **ROUGE-1 / ROUGE-2:** Overlap of individual words and word pairs.
- **ROUGE-L:** Longest common subsequence of words.
- **ROUGE-Lsum:** Summary-level ROUGE-L, accounting for sentence boundaries.

The `compute_rouge()` helper uses `rouge_score`: `use_aggregator=True` returns mean F1 scores; `False` returns one score per article.

In [23]:
from rouge_score import rouge_scorer

ROUGE_TYPES = ["rouge1", "rouge2", "rougeL", "rougeLsum"]

reference_sentence = "The quick brown fox jumps over the lazy dog."

candidate_sentence = "The quick brown fox jumps over the lazy dog."

scorer = rouge_scorer.RougeScorer(
    ROUGE_TYPES,
    use_stemmer=True
)

scorer.score(reference_sentence, candidate_sentence)

{'rouge1': Score(precision=1.0, recall=1.0, fmeasure=1.0),
 'rouge2': Score(precision=1.0, recall=1.0, fmeasure=1.0),
 'rougeL': Score(precision=1.0, recall=1.0, fmeasure=1.0),
 'rougeLsum': Score(precision=1.0, recall=1.0, fmeasure=1.0)}

In [24]:
candidate_sentence =  "A fast brown fox jumps over a lazy dog."

scorer = rouge_scorer.RougeScorer(
    ROUGE_TYPES,
    use_stemmer=True
)

scorer.score(reference_sentence, candidate_sentence)

{'rouge1': Score(precision=0.6666666666666666, recall=0.6666666666666666, fmeasure=0.6666666666666666),
 'rouge2': Score(precision=0.5, recall=0.5, fmeasure=0.5),
 'rougeL': Score(precision=0.6666666666666666, recall=0.6666666666666666, fmeasure=0.6666666666666666),
 'rougeLsum': Score(precision=0.6666666666666666, recall=0.6666666666666666, fmeasure=0.6666666666666666)}

In [25]:
candidate_sentence =  "The company reported strong quarterly earnings after market close."

scorer = rouge_scorer.RougeScorer(
    ROUGE_TYPES,
    use_stemmer=True
)

scorer.score(reference_sentence, candidate_sentence)

{'rouge1': Score(precision=0.1111111111111111, recall=0.1111111111111111, fmeasure=0.1111111111111111),
 'rouge2': Score(precision=0.0, recall=0.0, fmeasure=0.0),
 'rougeL': Score(precision=0.1111111111111111, recall=0.1111111111111111, fmeasure=0.1111111111111111),
 'rougeLsum': Score(precision=0.1111111111111111, recall=0.1111111111111111, fmeasure=0.1111111111111111)}

In [26]:
print(summary_txt)

print(ref_txt)

researchers at antwerp university in belgium, belgium, permeated a bookshop with the perfume of chocolate . they found customers were 3.5 times more likely to buy romance novels when they smelt chocolate . the government funded study also found the scent encouraged customers to browse through titles .
belgium scientists flooded a bookshop with a chocolatey smell to find customers were almost six times more likely to buy a romance novel . the government-funded study said the scent does not have a big effect on making people want to purchase travel, crime, business or comic books . researchers at antwerp university said the sweet smell encouraged customers to browse and . sales of all books rose during the experiment .


In [27]:
import re
import numpy as np

def prep_for_rouge_lsum(text):
    sentences = re.split(r"(?<=[.!?])\s+|\n+", text.strip())
    return "\n".join(sentence for sentence in sentences if sentence)


def compute_rouge(predictions, references, use_aggregator=True):
    scorer = rouge_scorer.RougeScorer(
        ROUGE_TYPES,
        use_stemmer=True
    )

    scores = {rouge_type: [] for rouge_type in ROUGE_TYPES}

    for prediction, reference in zip(predictions, references):
        result = scorer.score(
            prep_for_rouge_lsum(reference),
            prep_for_rouge_lsum(prediction)
        )

        for rouge_type in ROUGE_TYPES:
            scores[rouge_type].append(result[rouge_type].fmeasure)

    if use_aggregator:
        return {
            rouge_type: float(np.mean(values))
            for rouge_type, values in scores.items()
        }

    return scores

In [28]:
summary_result = compute_rouge(
    predictions=[summary_txt],
    references=[ref_txt],
)

summary_result

{'rouge1': 0.5043478260869565,
 'rouge2': 0.3185840707964602,
 'rougeL': 0.4,
 'rougeLsum': 0.5043478260869565}

### ROUGE comparison: DistilBART vs. instruction-tuned FLAN-T5

Generate two summaries of the same news article and compare their ROUGE scores against the same reference. This is an illustration, not a benchmark; different prompt lengths and decoding choices can affect the scores.

In [29]:
comparison_index = 8
comparison_article = cleaned_df.iloc[comparison_index]["Content"]
comparison_reference = cleaned_df.iloc[comparison_index]["Summary"]

distilbart_prediction = summarize(
    comparison_article, model_name="sshleifer/distilbart-cnn-12-6"
)[0]["summary_text"]

flan_prediction = flan_summarize(
    comparison_article, "Summarize this news article in two or three factual sentences."
)

In [30]:
model_names = ["DistilBART", "FLAN-T5-base"]
predictions = [distilbart_prediction, flan_prediction]

rouge_details = compute_rouge(
    predictions=predictions,
    references=[comparison_reference] * len(predictions),
    use_aggregator=False,
)

comparison_scores = pd.DataFrame({
    "Model": model_names,
    "ROUGE-1": rouge_details["rouge1"],
    "ROUGE-2": rouge_details["rouge2"],
    "ROUGE-Lsum": rouge_details["rougeLsum"],
    "Prediction": predictions,
})

display(comparison_scores.round(3))
print("REFERENCE SUMMARY:\n", comparison_reference)

,Model,ROUGE-1,ROUGE-2,ROUGE-Lsum,Prediction
0,DistilBART,0.504,0.319,0.504,"Researchers at antwerp university in belgium, Belgium, permeated a bookshop with the perfume of chocolate . They found customers were 3.5 times more likely to buy romance novels when they smelt chocolate . The government funded study also found the scent encouraged customers to browse through titles ."
1,FLAN-T5-base,0.330,0.084,0.289,"'The smell of chocolate makes people hungry to buy romance novels, but it does not necessarily tempt people to buy grittier genres such as crime or business books'"


REFERENCE SUMMARY:
 belgium scientists flooded a bookshop with a chocolatey smell to find customers were almost six times more likely to buy a romance novel . the government-funded study said the scent does not have a big effect on making people want to purchase travel, crime, business or comic books . researchers at antwerp university said the sweet smell encouraged customers to browse and . sales of all books rose during the experiment .


---

## Summarize 50 news articles and find the best and worst results

Restore the original activity: generate a summary for each of the first 50 articles using **DistilBART**, compare with the dataset's reference summary using ROUGE, and inspect the lowest- and highest-scoring examples.

The cached DistilBART model is reused for all 50 articles. This will still take some time, especially on CPU.

In [32]:
articles_txt = cleaned_df['Content']

articles_txt.sample(2)

,Content
4353,"by . ruth styles . published: . 09:09 est, 11 september 2013 . | . updated: . 07:26 est, 16 september 2013 . is your name bob, jim, ron or bill? if so, you're likely to be a man in demand after new research revealed that having a shortened name increases your appeal among the opposite sex. women who call themselves jen, val, debs and jo are also set to benefit as the effect is the same regardl..."
3347,"by . adrian durham . follow talksportdrive . i have just returned from brazil but not before i witnessed the dejection of a nation almost in mourning after neymar broke a bone in his back. i’ve heard some stupid things in my time, but for people to blame the referee for that injury is utter nonsense. it wasn’t an under 11s game – these are grown men. and one professional footballer decided to ..."


Use `tqdm` to show a progress bar while we summarize the first 50 articles and
collect the candidate summaries.

In [33]:
from tqdm.auto import tqdm

candidate_summaries = []

for text in tqdm(articles_txt[:50], desc="Summarizing articles"):
    candidate = summarize(text, model_name="sshleifer/distilbart-cnn-12-6")
    candidate_summaries.append(candidate[0]["summary_text"])

print(f"Generated {len(candidate_summaries)} summaries")

Summarizing articles:   0%|          | 0/50 [00:00<?, ?it/s]

Generated 50 summaries


In [34]:
summaries_txt = cleaned_df['Summary']

summaries_txt.sample(2)

,Summary
4724,"sarah young remembers being told the devastating news that nothing more could be done for her five-year-old son, ollie."
4110,"hong kong fastest for peak connection speeds, past 50 mbps . south korea, japan also lead global rankings compiled by akamai ."


In [35]:
result_agg = compute_rouge(
    predictions=candidate_summaries,
    references=list(summaries_txt[:50])
)

result_agg

{'rouge1': 0.37524164443023955,
 'rouge2': 0.17708631753451884,
 'rougeL': 0.27893484401729607,
 'rougeLsum': 0.34640155320393234}

In [36]:
result_unagg = compute_rouge(
    predictions=candidate_summaries,
    references=list(summaries_txt[:50]),
    use_aggregator=False,
)

result_unagg

{'rouge1': [0.3829787234042554,
  0.41509433962264153,
  0.5084745762711864,
  0.5476190476190477,
  0.27199999999999996,
  0.3779527559055118,
  0.2857142857142857,
  0.5420560747663552,
  0.5043478260869565,
  0.38636363636363635,
  0.36893203883495146,
  0.5168539325842696,
  0.4489795918367347,
  0.09677419354838708,
  0.3188405797101449,
  0.3695652173913043,
  0.1818181818181818,
  0.3783783783783784,
  0.7580645161290323,
  0.2086330935251799,
  0.36111111111111105,
  0.22222222222222224,
  0.273972602739726,
  0.20588235294117646,
  0.42696629213483145,
  0.3333333333333333,
  0.22222222222222224,
  0.3373493975903615,
  0.1276595744680851,
  0.3576158940397351,
  0.4117647058823529,
  0.5794392523364486,
  0.18823529411764706,
  0.48888888888888893,
  0.4864864864864865,
  0.2527075812274368,
  0.1739130434782609,
  0.4166666666666667,
  0.25,
  0.1518987341772152,
  0.5192307692307692,
  0.20338983050847456,
  0.628099173553719,
  0.5614035087719298,
  0.7714285714285715,
  0

Find the best and worst summaries by ROUGE-Lsum (min and max indices).

In [37]:
result_unagg_rsum = np.array(result_unagg["rougeLsum"])

max_arg = np.argmax(result_unagg_rsum)
min_arg = np.argmin(result_unagg_rsum)

max_arg, min_arg

(np.int64(44), np.int64(16))

**Worst** candidate summary and its reference:

In [38]:
candidate_summaries[min_arg]

'Manxman kneen, 28, has been ruled out after breaking his arm in a mountain biking crash . mar-train boss tim martin revealed that kneen had undergone surgery but remains hopeful that the manx rider will be fit for next month isle of man tt .'

In [39]:
summaries_txt[min_arg]

'former motorgp star jeremy mcwilliams will replace injured dan kneen in the mar-train yamaha team at this week north west 200.'

**Best** candidate summary and its reference:

In [40]:
candidate_summaries[max_arg]

'janelle duncan-bailey, 25, went missing in south london in the early hours of wed Wednesday . Her body was found yesterday afternoon in mayfield crescent, thornton heath . Jerome mcdonald, 30, has been charged with her murder and will appear in court tomorrow .'

In [41]:
summaries_txt[max_arg]

'janelle duncan-bailey went missing in south london early on wednesday . her body was found yesterday in thornton heath . jerome mcdonald, 30, has been charged with her murder .'

In [43]:
act_vs_pred_summaries_df = pd.DataFrame(
    list(zip(candidate_summaries, summaries_txt[:50])),
    columns=["Predicted_Summaries", "Reference_summaries"],
)

act_vs_pred_summaries_df.head(5)

,Predicted_Summaries,Reference_summaries
0,victor valdes enjoyed a meal out at san carlo on thursday night hours after completing his move . The former Barcelona goalkeeper will battle with current no 1 david de gea at old trafford . Fellow spanish-speakers angel di maria and marcos rojo also dined out at the popular Italian restaurant .,victor valdes posed with a fan outside san carlo on thursday night . manchester united keeper took part in training session next morning . he will battle with david de gea at old trafford after signing 18-month deal . click here for more manchester united news .
1,Aviation industry is often seen as one of the bad guys in the climate change debate . Aviation is responsible for around 3.5 percent of global greenhouse emissions . jet republic's ceo says it would be a mistake to let green initiatives in the airline industry fall by the wayside . The airline is 100 percent carbon neutral from launch .,jet republic has teamed up with climatecare to become carbon neutral . airline chose fleet of learjet 60 xr partly because of its fuel efficiency . aviation industry is responsible for 3.5 percent of global carbon emissions . breeze: airline industry unfairly criticized for failing to address climate change .
2,"London police to investigate claims that british secret services were involved in rendition of two men to libya . One of the cases involves abdul hakim belhaj, a former leader of the libyan islamic fighting group . The second is libyan citizen saami al-saadi, officials confirm . The government stands firmly against torture and cruel, inhuman and degrading treatment or punishment .","new: the uk government stands firmly against torture and will cooperate fully, it says . claims made by two libyan men are so serious they should be investigated now, police say . abdul hakim belhaj and saami al-saadi say british officials were involved in their rendition . the head of mi6 says the intelligence agency will cooperate fully with the police ."
3,"The blast occurred at a storage tank that was undergoing maintenance, police and fire officials said . It was not immediately clear what caused the explosion, police chief superintendent gwyn thomas said . One contractor who was injured in the explosion was being treated for serious injuries, the refinery general manager said .","an explosion occurrs at a storage tank that was undergoing maintenance, officials say . chevron refinery manager identifies the dead and wounded as contractors . authorities say it is not immediately clear what caused the explosion ."
4,"Family believes kristi and benjamin strack killed children, committed suicide . The stracks were found dead by an older son, janson mcgee, with cups of red liquid next to their bodies, on sept. 27 . The children – benson, 14, emery, 12, and zion, 11 – were found lying on and around the bed, covered with blankets up to their necks, with empty bottles of liquid methadone and boxes of cold and fl...","– police suspect foul play and poisoning in the deaths of a utah family of five, but they haven't laid out an official explanation of what they think transpired. relatives, however, tell people that it clear parents benjamin and kristi strack poisoned their three children and then committed suicide. the two suffered from mental health problems that were worsened by drug addictions, they say. i..."
